# Early Learning Support: Final Evaluation

[Open in Colab](https://colab.research.google.com/github/Junghwan0516/early-learning-support/blob/main/03_final_evaluation.ipynb)

**Status: prepared, not evaluated.**

Use this notebook only after accepting the frozen method and experiment settings. Inspecting the final-test result must not trigger retuning on the same test. The method is fitted on the original training cohort, matching the validation experiment; no train-plus-validation refit is silently introduced.

The 2014J presentation is a held-out cohort, not proof of institution-wide or prospective deployment performance.

In [ ]:
from pathlib import Path
import urllib.request
import sys

repository_root = 'https://raw.githubusercontent.com/Junghwan0516/early-learning-support/main/'
for filename in ['els.py', 'prepare_data.py']:
    if not Path(filename).exists():
        with urllib.request.urlopen(repository_root + filename, timeout=60) as response:
            Path(filename).write_bytes(response.read())
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import els
els.style()

In [ ]:
RUN_FINAL_TEST = False

if RUN_FINAL_TEST:
    import json
    selection_path = Path('reports/frozen_selection.json')
    if not selection_path.exists():
        selection_path.parent.mkdir(exist_ok=True)
        with urllib.request.urlopen(repository_root + str(selection_path), timeout=60) as response:
            selection_path.write_bytes(response.read())
    selection = json.loads(selection_path.read_text())
    parts, audit = els.split_data(els.load_data())
    if selection['features'] != els.FEATURES:
        raise ValueError('The feature contract has changed since selection was frozen.')
    if selection['family'] == 'Low-activity rule':
        scores = -parts['test'].total_clicks.to_numpy()
    else:
        model = els.model_candidates()[(selection['family'], selection['config'])]
        model.fit(parts['train'][els.FEATURES], parts['train'].target)
        scores = els.scores_for(model, parts['test'][els.FEATURES])
    metrics = els.ranking_metrics(parts['test'].target, scores)
    result = {'selection': selection, 'final_test_metrics': metrics}
    Path('reports/final_test_metrics.json').write_text(json.dumps(result, indent=2))
    print(result)
else:
    print('Final evaluation is disabled. No test scores have been computed.')

## Reporting after execution

Report the fixed method's test performance, the review budget and captured withdrawals, and limitations. Compare validation and test estimates without changing settings in response. If the test result motivates a new method, define a new development/evaluation design rather than treating the already inspected cohort as untouched.